## GPU Computing - Introduction

### ****Exercise 1:** Hello GPU**

**Run basic `basic_test.py` (attached on Moodle) and confirm your GPU appears.**

In [1]:
%run basic_test.py

Device: gfx1035
  Vendor:  Advanced Micro Devices, Inc.
  OpenCL:  OpenCL 2.0 
  Compute units: 6



/opt/conda/envs/nsc/lib/python3.11/site-packages/pyopencl/__init__.py:578: UserWarning: PyOpenCL compiler caching failed with an exception:
[begin exception]
Traceback (most recent call last):
  File "/opt/conda/envs/nsc/lib/python3.11/site-packages/pyopencl/cache.py", line 514, in create_built_program_from_source_cached
    _create_built_program_from_source_cached(
  File "/opt/conda/envs/nsc/lib/python3.11/site-packages/pyopencl/cache.py", line 424, in _create_built_program_from_source_cached
    src = src + b"\n\n__constant int pyopencl_defeat_cache_%s = 0;" % (
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: %b requires a bytes-like object, or an object that implements __bytes__, not 'str'
[end exception]
  lambda: create_built_program_from_source_cached(


Kernel output: [ 1.  4.  9. 16.]
All elements close? True


We can also use clinfo for this.

In [2]:
!clinfo --list

Platform #0: AMD Accelerated Parallel Processing
 `-- Device #0: gfx1035
Platform #1: Intel(R) OpenCL
 `-- Device #0: AMD Ryzen 7 PRO 6850U with Radeon Graphics     


- **You should see your device listed — e.g. `Apple [’Apple M1 Max’]` or `NVIDIA CUDA [’GeForce RTX 3080’]`**

I see my iGPU "gfx1035" 

- **Only `Portable Computing Language [’cpu’]`? The conda package found a CPU-only fallback:**
    - **With your NSC environment active: `mamba remove pyopencl` then `pip install pyopencl`**

Noted! I am using my GPU. No generic CPU fallback.

- **Note your device name, compute unit count, and whether fp64 is supported**

In [3]:
import pyopencl as cl

platforms = cl.get_platforms()
for platform in platforms:
    print(f"Platform:\t{platform.name}")
    devices = platform.get_devices()
    for device in devices:
        ctx = cl.Context(devices=[device])
        dev = ctx.devices[0]

        print(f"Device Name:\t{dev.name}")
        print(f"Compute Unit Count:\t{dev.max_compute_units}")
        print(f"FP64 Support:\t{'cl_khr_fp64' in dev.extensions}")
        print()


Platform:	AMD Accelerated Parallel Processing
Device Name:	gfx1035
Compute Unit Count:	6
FP64 Support:	True

Platform:	Intel(R) OpenCL
Device Name:	AMD Ryzen 7 PRO 6850U with Radeon Graphics     
Compute Unit Count:	16
FP64 Support:	True



Both of my devices support float64 extension.

****Done?** Device visible → move on to E2**

### ****Exercise 2:** Vector Sum**

****Task:** Start from `opencl_template.py` (on Moodle) and change the kernel to compute `result[i] = a[i] + b[i]` for two float32 arrays of length *N* = 50 000. Verify with NumPy.**

```python
# From slide 6

KERNEL_SRC = """
__kernel void vector_sum(
    __global const float *a,
    __global const float *b,
    __global float *result)
{
    int gid = get_global_id(0);
    result[gid] = ???;
    // fill this in
}
"""
# Replace the kernel in opencl_template.py with the above.
# Launch (already in template):
prog.vector_sum(queue, (N,), None, a_dev, b_dev, r_dev)
queue.finish()
print(np.allclose(r_host, a_host + b_host)) # True
```

Noted! I will be using the opencl template inside of this notebook.

In [4]:
# A direct copy of opencl_template.py

import time
import numpy as np
import pyopencl as cl

VEC_SIZE = 50_000

# --- Step 1: create context and command queue ---
ctx   = cl.create_some_context(interactive=False)
queue = cl.CommandQueue(ctx)
print(f"Device: {ctx.devices[0].name}")

# --- Step 2: prepare host arrays ---
a_host      = np.random.rand(VEC_SIZE).astype(np.float32)
b_host      = np.random.rand(VEC_SIZE).astype(np.float32)
result_host = np.empty_like(a_host)

# --- Step 3: allocate device buffers and copy input data ---
mf       = cl.mem_flags
a_dev    = cl.Buffer(ctx, mf.READ_ONLY  | mf.COPY_HOST_PTR, hostbuf=a_host)
b_dev    = cl.Buffer(ctx, mf.READ_ONLY  | mf.COPY_HOST_PTR, hostbuf=b_host)
res_dev  = cl.Buffer(ctx, mf.WRITE_ONLY, a_host.nbytes)

# --- Step 4: compile the kernel ---
# To load from a separate file instead: KERNEL_SRC = open("kernel.cl").read()
KERNEL_SRC = """
__kernel void sum(
    __global const float *a,
    __global const float *b,
    __global       float *result)
{
    int gid = get_global_id(0);
    result[gid] = a[gid] + b[gid];
}
"""
prog = cl.Program(ctx, KERNEL_SRC).build()

# --- Step 5: launch the kernel ---
t0 = time.perf_counter()
prog.sum(queue, a_host.shape, None, a_dev, b_dev, res_dev)
queue.finish()
elapsed = time.perf_counter() - t0

# --- Step 6: copy result back to host ---
cl.enqueue_copy(queue, result_host, res_dev)
queue.finish()

# Verify and report
print(f"Elapsed:  {elapsed*1000:.3f} ms")
print(f"Correct:  {np.allclose(result_host, a_host + b_host)}")


Device: gfx1035
Elapsed:  2.304 ms
Correct:  True


- **The kernel body is a single line — see the code example**

Noted! I have seen it.

- **Check: `assert np.allclose(result_host, a_host + b_host)`**

Done! Please see the code above from the template. The result is correct.

****Questions — understand these before moving on:****

- **What value does `get_global_id(0)` return for work-item 42 if `global_size = (50000,)`?**

In [5]:
VEC_SIZE = 50_000

# --- Step 1: create context and command queue ---
ctx = cl.create_some_context(interactive=False)
queue = cl.CommandQueue(ctx)
print(f"Device: {ctx.devices[0].name}")

# --- Step 2: prepare host arrays ---
a_host = np.random.rand(VEC_SIZE).astype(np.float32)
b_host = np.random.rand(VEC_SIZE).astype(np.float32)
result_host = np.empty_like(a_host)

# --- Step 3: allocate device buffers and copy input data ---
mf = cl.mem_flags
a_dev = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=a_host)
b_dev = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=b_host)
res_dev = cl.Buffer(ctx, mf.WRITE_ONLY, a_host.nbytes)

# --- Step 4: compile the kernel ---
# To load from a separate file instead: KERNEL_SRC = open("kernel.cl").read()
KERNEL_SRC = """
__kernel void sum(
    __global const float *a,
    __global const float *b,
    __global       float *result)
{
    int gid = get_global_id(0);

    // I am guessing this is what is asked for
    if (gid == 42) 
        printf("get_global_id(0) returned: %d", gid);
    result[gid] = a[gid] + b[gid];
}
"""
prog = cl.Program(ctx, KERNEL_SRC).build()

# --- Step 5: launch the kernel ---
t0 = time.perf_counter()
prog.sum(queue, a_host.shape, None, a_dev, b_dev, res_dev)
queue.finish()
elapsed = time.perf_counter() - t0

# --- Step 6: copy result back to host ---
cl.enqueue_copy(queue, result_host, res_dev)
queue.finish()

# Verify and report
print(f"Elapsed:  {elapsed*1000:.3f} ms")
print(f"Correct:  {np.allclose(result_host, a_host + b_host)}")

Device: gfx1035
get_global_id(0) returned: 42
Elapsed:  2.684 ms
Correct:  True


- **What happens if you launch with `(49999,)` instead of `(N,)`?**

In [6]:
VEC_SIZE = 49999

# --- Step 1: create context and command queue ---
ctx = cl.create_some_context(interactive=False)
queue = cl.CommandQueue(ctx)
print(f"Device: {ctx.devices[0].name}")

# --- Step 2: prepare host arrays ---
a_host = np.random.rand(VEC_SIZE).astype(np.float32)
b_host = np.random.rand(VEC_SIZE).astype(np.float32)
result_host = np.empty_like(a_host)

# --- Step 3: allocate device buffers and copy input data ---
mf = cl.mem_flags
a_dev = cl.Buffer(ctx, mf.READ_WRITE | mf.COPY_HOST_PTR, hostbuf=a_host)
b_dev = cl.Buffer(ctx, mf.READ_WRITE | mf.COPY_HOST_PTR, hostbuf=b_host)
res_dev = cl.Buffer(ctx, mf.READ_WRITE, a_host.nbytes)

# --- Step 4: compile the kernel ---
# To load from a separate file instead: KERNEL_SRC = open("kernel.cl").read()
KERNEL_SRC = """
__kernel void sum(
    __global const float *a,
    __global const float *b,
    __global       float *result)
{
    int gid = get_global_id(0);

    // I am guessing this is what is asked for
    if (gid == 42) 
        printf("get_global_id(0) returned: %d", gid);
    result[gid] = a[gid] + b[gid];
}
"""
prog = cl.Program(ctx, KERNEL_SRC).build()

# --- Step 5: launch the kernel ---
t0 = time.perf_counter()
prog.sum(queue, a_host.shape, None, a_dev, b_dev, res_dev)
queue.finish()
elapsed = time.perf_counter() - t0

# --- Step 6: copy result back to host ---
cl.enqueue_copy(queue, result_host, res_dev)
queue.finish()

# Verify and report
print(f"Elapsed:  {elapsed*1000:.3f} ms")
print(f"Correct:  {np.allclose(result_host, a_host + b_host)}")

Device: gfx1035
get_global_id(0) returned: 42
Elapsed:  2.185 ms
Correct:  True


We get the same global work-item id.

- **Why must the result buffer use `WRITE_ONLY` and inputs use `READ_ONLY`?**

The result buffer must be `WRITE_ONLY` since we are writing to it in the kernel.

The input buffers must be `READ_ONLY` since we are reading from them via the kernel.

Marking the buffers as `READ_WRITE` would also be fine, but less optimal.

****Done?** `np.allclose` passes → discuss with a neighbour**

`np.allclose` does pass **✓**

TODO: add optional exercise if time allows for it.